####**1. Loading Dataset:**

In [ ]:
# Importing Libraries:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import zipfile
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

from joblib import Parallel, delayed
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline

from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif, SelectFromModel, RFE
from imblearn.over_sampling import SMOTE

from sklearn.linear_model import LogisticRegression, LassoCV,Lasso

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import StackingClassifier

from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report

from sklearn.model_selection import RandomizedSearchCV
import optuna


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Loading Cleaned Dataset:
df = pd.read_csv('/content/drive/My Drive/EV_CleanedDataset.csv')
# Look at the Dataset:
df.head()


,account,premise_desc_code,size,customer_type,heating_type,heating_group,last_yr_kwh_usage,last_yr_avg_kwh_usage,geo_area_code,city,...,children_desc_code,can_afford_ev,est_rate_code,recently_bought_home,est_social_media_income,est_alternative_financing,EVFlag,energy_efficiency_index,affordability_index,sustainability_interest_index
0,1,DU,SMALL,MULTI FAMILY- ELECTRIC,ELECTRIC BASEBOARD,Electric,16279.0,1281.0,0.0,0.0,...,0.0,0.68,0.0,0.0,0.0,0.86,0.0,10.048765,0.00,0.0
1,2,SF,MED,SINGLE FAMILY - OTHER HEAT,GAS FURNACE,Non-Electric,48531.0,6036.0,1.0,1.0,...,1.0,0.76,0.0,0.0,1.0,0.87,0.0,12.711105,0.24,0.0
2,3,SF,MED,SINGLE FAMILY - ELECTRIC,ELECTRIC HEAT PUMP,Electric,9801.0,846.0,2.0,2.0,...,0.0,0.96,0.0,0.0,0.0,0.82,0.0,3.296670,0.08,1.0
3,4,SF,MED,SINGLE FAMILY - ELECTRIC,ELECTRIC HEAT PUMP,Electric,9801.0,846.0,2.0,2.0,...,0.0,0.96,0.0,0.0,0.0,0.82,0.0,3.296670,0.08,1.0
4,5,SF,MED,SINGLE FAMILY - ELECTRIC,ELECTRIC HEAT PUMP,Electric,7408.0,602.0,2.0,2.0,...,0.0,0.96,0.0,0.0,0.0,0.82,0.0,2.491759,0.08,1.0


In [ ]:
# Look at the Shape
df.shape

(393854, 69)

In [ ]:
# Look at the columns
df.columns

Index(['account', 'premise_desc_code', 'size', 'customer_type', 'heating_type',
       'heating_group', 'last_yr_kwh_usage', 'last_yr_avg_kwh_usage',
       'geo_area_code', 'city', 'zip', 'substation', 'circuit', 'move_in_date',
       'est_square_footage', 'est_year_built', 'has_solar', 'est_adults_total',
       'est_children_age_ranges', 'est_nationality', 'dwelling_type',
       'is_economically_stable', 'est_education_level', 'uses_email',
       'budget_concious', 'values_convenience', 'values_sustainability',
       'values_energy_efficiency', 'values_investments', 'behavior_segment',
       'est_generations_present', 'est_room_count', 'home_exterior_type',
       'home_heat_source', 'heating_cooling', 'est_length_lived_in_property',
       'est_home_value', 'est_owner_or_renter', 'home_owner_type',
       'home_has_pool', 'property_type', 'home_bought_date',
       'home_roof_material', 'est_square_footage_range',
       'est_year_built_range', 'est_household_size',
       'es

####**2. Reduce Features based on Correlationship**






In [ ]:
# Look at the correlation between EVFlag and all columns:
df_encoded = df.copy()

# Convert categorical columns to numeric using label encoding
for col in df_encoded.select_dtypes(include=['object']).columns:
    df_encoded[col] = df_encoded[col].astype('category').cat.codes

df_encoded.corr()["EVFlag"].sort_values(ascending=False)[:20]

,EVFlag
EVFlag,1.000000
size,0.295899
est_year_built,0.133390
move_in_date,0.132608
est_alternative_financing,0.132038
values_convenience,0.105357
est_lifestage,0.099972
est_placement_segment,0.084773
est_net_worth,0.082652
uses_email,0.072776


In [ ]:
# Look at the absolute correlation between EVFlag and all columns:
np.abs(df_encoded.corr()["EVFlag"]).sort_values(ascending=False)[:20]

,EVFlag
EVFlag,1.000000
premise_desc_code,0.314151
size,0.295899
dwelling_type,0.274890
customer_type,0.244217
est_owner_or_renter,0.175324
home_owner_type,0.136299
est_year_built,0.133390
move_in_date,0.132608
est_alternative_financing,0.132038


In [ ]:
# Calculate the correlation matrix
corr_matrix = df_encoded.corr()
threshold = 0.8

# Find pairs of columns with correlation greater than the threshold
high_corr_pairs = []

for col in corr_matrix.columns:
    for row in corr_matrix.index:
        if abs(corr_matrix.loc[row, col]) > threshold and row != col:
            high_corr_pairs.append((row, col, corr_matrix.loc[row, col]))

# Remove duplicate pairs:
high_corr_pairs = list(set([tuple(sorted(pair[:2])) + (pair[2],) for pair in high_corr_pairs]))

for pair in high_corr_pairs:
    print(f"{pair[0]} with {pair[1]}: {pair[2] * 100:.0f}%")


last_yr_avg_kwh_usage with last_yr_kwh_usage: 98%
sustainability_interest_index with values_sustainability: 98%
premise_desc_code with size: -85%


In [ ]:
# REmove Hicorrelation and Useless columns:
df.drop(['last_yr_kwh_usage','values_sustainability','account'],axis=1,inplace=True)

####**3. Converting NonNumeric to Numeric Columns\:**





In [ ]:
# Extract object columns again:
nonnumeric_columns = df.select_dtypes('object')
nonnumeric_columns.columns

Index(['premise_desc_code', 'size', 'customer_type', 'heating_type',
       'heating_group', 'move_in_date', 'home_bought_date'],
      dtype='object')

In [ ]:
# Define the columns to encode
one_hot_columns = ['premise_desc_code', 'customer_type', 'heating_type', 'heating_group']
ordinal_columns = ['size'] # Will Encoding
date_columns = ['move_in_date', 'home_bought_date']

# 1. One-Hot Encoding for Categorical Columns
df_one_hot = pd.get_dummies(df, columns=one_hot_columns, drop_first=True)

# 2. Ordinal Encoding for Ordered Categories (like 'size')
size_mapping = {'SMALL': 1, 'MED': 2, 'LARGE': 3, 'XL': 4, 'Unassigned': 0}
df_one_hot['size'] = df_one_hot['size'].map(size_mapping)

# 3. Handle Date Columns by Extracting Features (e.g., days since move-in/home bought)

# Convert move_in_date and home_bought_date to datetime
df_one_hot['move_in_date'] = pd.to_datetime(df_one_hot['move_in_date'], errors='coerce')
df_one_hot['home_bought_date'] = pd.to_datetime(df_one_hot['home_bought_date'], errors='coerce')

# Calculate 'days since' for move_in_date and home_bought_date
df_one_hot['days_since_move_in'] = (pd.to_datetime('today') - df_one_hot['move_in_date']).dt.days
df_one_hot['days_since_home_bought'] = (pd.to_datetime('today') - df_one_hot['home_bought_date']).dt.days

# Extract year, month, and day from the dates
df_one_hot['move_in_year'] = df_one_hot['move_in_date'].dt.year
df_one_hot['move_in_month'] = df_one_hot['move_in_date'].dt.month
df_one_hot['move_in_day'] = df_one_hot['move_in_date'].dt.day

df_one_hot['home_bought_year'] = df_one_hot['home_bought_date'].dt.year
df_one_hot['home_bought_month'] = df_one_hot['home_bought_date'].dt.month
df_one_hot['home_bought_day'] = df_one_hot['home_bought_date'].dt.day

# Drop the original 'move_in_date' and 'home_bought_date' columns if you don't need them
df_one_hot.drop(columns=['move_in_date', 'home_bought_date'], inplace=True)

#### **4. Split Dataset:**

In [ ]:
# Check class distribution
print(df_one_hot['EVFlag'].value_counts(normalize=True))


EVFlag
0.0    0.89831
1.0    0.10169
Name: proportion, dtype: float64


In [ ]:
# Split Dataset
X = df_one_hot.drop(columns=['EVFlag'],axis=1)  # Drop the target variable
y = df_one_hot['EVFlag']

# Split into 80% train, 20% test (stratify: ensures both the train and test sets have the same EV vs. non-EV ratio as the original dataset.)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


####**5. Feature Selelection and Applying Various Prediction Models:**




In [ ]:
def preprocess_data(X_train, X_test, y_train):
    # Store original feature names before transformations
    original_columns = X_train.columns

    # Standardization
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # SMOTE (Synthetic Minority Over-sampling Technique):creates new synthetic examples of the minority class (EV owners)
    #instead of just duplicating existing ones.# Ex:If you have an EV owner at (x=3, y=5) and another at (x=4, y=6),
    #SMOTE might create a new synthetic EV sample at (x=3.5, y=5.5).
    smote = SMOTE(sampling_strategy=0.25, random_state=42) # Make EVs 25% of total
    X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

    # Feature selection (Mutual Information)
    selector_mi = SelectKBest(score_func=mutual_info_classif, k=50)
    X_train_mi = selector_mi.fit_transform(X_train_res, y_train_res)
    X_test_mi = selector_mi.transform(X_test_scaled)
    selected_features_mi = original_columns[selector_mi.get_support()]
    print(f"Selected columns with SelectKBest and Mutual Info: {list(selected_features_mi)}")

    # Feature selection with Random Forest
    rf_selector = SelectFromModel(RandomForestClassifier(n_estimators=100, random_state=42), threshold=-np.inf, max_features=40)
    X_train_rf = rf_selector.fit_transform(X_train_mi, y_train_res)
    X_test_rf = rf_selector.transform(X_test_mi)
    # Update feature names after Random Forest selection
    selected_features_rf = [selected_features_mi[i] for i in range(len(selected_features_mi)) if rf_selector.get_support()[i]]
    print(f"Selected columns with Random Forest: {list(selected_features_rf)}")

    # Feature selection with Lasso
    lasso_selector = SelectFromModel(Lasso(alpha=0.00001,random_state=42), max_features=35)
    X_train_lasso = lasso_selector.fit_transform(X_train_rf, y_train_res)
    X_test_lasso = lasso_selector.transform(X_test_rf)
    # Ensure 35 features are selected with Lasso
    selected_features_lasso = [selected_features_rf[i] for i in range(len(selected_features_rf)) if lasso_selector.get_support()[i]]
    print(f"Selected columns with Lasso: {list(selected_features_lasso)}")

    # Feature selection with RFE
    rfe_selector = RFE(estimator=LogisticRegression(max_iter=1000, random_state=42), n_features_to_select=30)
    X_train_rfe = rfe_selector.fit_transform(X_train_lasso, y_train_res)
    X_test_rfe = rfe_selector.transform(X_test_lasso)
    # Ensure exactly 30 features are selected with RFE
    selected_features_rfe = [selected_features_lasso[i] for i in range(len(selected_features_lasso)) if rfe_selector.get_support()[i]]
    print(f"Selected columns with RFE: {list(selected_features_rfe)}")

    return X_train_rfe, X_test_rfe, y_train_res, y_train

# Preprocess the data once
X_train_processed, X_test_processed, y_train_res, y_train = preprocess_data(X_train, X_test, y_train)

Selected columns with SelectKBest and Mutual Info: ['size', 'last_yr_avg_kwh_usage', 'geo_area_code', 'city', 'zip', 'substation', 'circuit', 'est_square_footage', 'est_year_built', 'est_adults_total', 'est_nationality', 'dwelling_type', 'is_economically_stable', 'est_education_level', 'budget_concious', 'values_convenience', 'values_investments', 'behavior_segment', 'est_room_count', 'est_length_lived_in_property', 'est_home_value', 'est_owner_or_renter', 'est_square_footage_range', 'est_year_built_range', 'est_household_size', 'est_household_income_range', 'est_marital_status', 'likes_social_media', 'est_net_worth', 'est_lifestage', 'est_placement_group', 'est_placement_segment', 'est_main_segment', 'est_population', 'can_afford_ev', 'est_rate_code', 'est_social_media_income', 'est_alternative_financing', 'affordability_index', 'sustainability_interest_index', 'premise_desc_code_APT', 'premise_desc_code_SF', 'customer_type_MULTI FAMILY- ELECTRIC', 'days_since_move_in', 'days_since_ho

In [ ]:
# Function to train and evaluate a model
def train_and_evaluate(name, model):
    print(f"Training {name}...")

    # Fit the model
    model.fit(X_train_processed, y_train_res)

    # Make predictions
    y_pred = model.predict(X_test_processed)
    y_proba = model.predict_proba(X_test_processed)[:, 1]  # Probability for ROC AUC

    # Store evaluation metrics
    return name, {
        'Accuracy': accuracy_score(y_test, y_pred),
        'AUC-ROC': roc_auc_score(y_test, y_proba),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred)
    }

# Define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    'LightGBM': LGBMClassifier(random_state=42),
}

# Run models in parallel
results = dict(Parallel(n_jobs=4)(delayed(train_and_evaluate)(name, model) for name, model in models.items()))

# Convert results to DataFrame for comparison
results_df = pd.DataFrame(results).T
print(results_df.sort_values(by='AUC-ROC', ascending=False))


                     Accuracy   AUC-ROC  Precision    Recall  F1-Score
Random Forest        0.959846  0.919881   0.872560  0.708614  0.782087
XGBoost              0.946122  0.892336   0.836010  0.584894  0.688262
LightGBM             0.942326  0.880927   0.846908  0.528340  0.650727
Logistic Regression  0.889680  0.803958   0.445108  0.344195  0.388201


####**6. Hyperparameter Optimization:**

In [ ]:
# Feature importance:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
feature_importances = pd.Series(rf.feature_importances_, index=X_train.columns)
print(feature_importances.sort_values(ascending=False).head(20))  # Top 20 important features


est_year_built             0.076140
circuit                    0.052913
geo_area_code              0.048894
substation                 0.046507
zip                        0.042122
est_population             0.038234
city                       0.029023
days_since_move_in         0.024053
energy_efficiency_index    0.023890
last_yr_avg_kwh_usage      0.022769
est_square_footage         0.021693
est_placement_segment      0.018613
size                       0.018246
est_home_value             0.017503
move_in_day                0.016435
premise_desc_code_SF       0.016324
dwelling_type              0.016114
est_main_segment           0.015910
affordability_index        0.015331
can_afford_ev              0.014542
dtype: float64


In [ ]:
# RandomizedSearchCV

param_dist = {
    'n_estimators': [100, 300],
    'max_depth': [10, 20],
    'min_samples_split': [2, 10],
    'min_samples_leaf': [1, 2]
}

random_search = RandomizedSearchCV(RandomForestClassifier(random_state=42),
                                   param_distributions=param_dist,
                                   n_iter=10,  # Searches only 10 random combinations instead of all
                                   scoring='roc_auc',
                                   cv=3,
                                   n_jobs=4,
                                   random_state=42)

random_search.fit(X_train_processed, y_train_res)

print("Best parameters:", random_search.best_params_)
best_rf = random_search.best_estimator_


/usr/local/lib/python3.11/dist-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best parameters: {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': 20}


In [ ]:
# XGBoost Optimization (using Optuna):
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.3)
    }
    model = XGBClassifier(**params, random_state=42, use_label_encoder=False, eval_metric='logloss', n_jobs=4)
    model.fit(X_train_processed, y_train_res)
    y_proba = model.predict_proba(X_test_processed)[:, 1]
    return roc_auc_score(y_test, y_proba)

# Create and run the Optuna study
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=7)

# Access the best parameters
print("Best parameters:", study.best_trial.params)

# Initialize the best model with the best parameters found
best_xgb = XGBClassifier(**study.best_trial.params, random_state=42, use_label_encoder=False, eval_metric='logloss')


[I 2025-02-19 19:21:28,430] A new study created in memory with name: no-name-fdbf91ba-1f65-4c53-bdba-44889ce2ac04
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [19:21:29] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
[I 2025-02-19 19:21:37,564] Trial 0 finished with value: 0.8671355508558015 and parameters: {'n_estimators': 413, 'max_depth': 3, 'learning_rate': 0.06967265532136976}. Best is trial 0 with value: 0.8671355508558015.
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [19:21:38] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
[I 2025-02-19 19:21:50,564] Trial 1 finished with value: 0.8963992240518588 and parameters: {'n_estimators': 245, 'max_depth': 8, 'learning_rate': 0.021412233305287545}. Best is trial 1 with value: 0.8963992240518588.
/usr/local/lib/python3.11/d

Best parameters: {'n_estimators': 245, 'max_depth': 8, 'learning_rate': 0.021412233305287545}


In [ ]:
# Function to train and evaluate a model
def train_and_evaluate(name, model):
    print(f"Training {name}...")

    # Fit the model
    model.fit(X_train_processed, y_train_res)

    # Make predictions
    y_pred = model.predict(X_test_processed)
    y_proba = model.predict_proba(X_test_processed)[:, 1]  # Probability for ROC AUC

    # Store evaluation metrics
    return name, {
        'Accuracy': accuracy_score(y_test, y_pred),
        'AUC-ROC': roc_auc_score(y_test, y_proba),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred)
    }

# Define models
models = {
    'Random Forest': RandomForestClassifier(n_estimators=300, random_state=42,min_samples_split=2,min_samples_leaf=1,max_depth=20),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42, n_estimators=423, max_depth=10,learning_rate=0.025213396236200107),
    'stacked_model': StackingClassifier(estimators=[
    ('rf', RandomForestClassifier(n_estimators=300, random_state=42)),
    ('xgb', XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)),
    ('lgbm', LGBMClassifier(random_state=42))
], final_estimator=LogisticRegression())

}

# Run models in parallel
results = dict(Parallel(n_jobs=4)(delayed(train_and_evaluate)(name, model) for name, model in models.items()))

# Convert results to DataFrame for comparison
results_df = pd.DataFrame(results).T
print(results_df.sort_values(by='AUC-ROC', ascending=False))


               Accuracy   AUC-ROC  Precision    Recall  F1-Score
stacked_model  0.968859  0.942262   0.924652  0.755306  0.831444
Random Forest  0.962080  0.931962   0.938997  0.670662  0.782463
XGBoost        0.961521  0.926637   0.934544  0.668414  0.779387
